# Tests: `fasterai.analyze.sensitivity` (source `nbs/analyze/sensitivity.ipynb`)

In [ ]:
from fastcore.test import *
import torch
from fasterai.analyze.sensitivity import *
from fasterai.core.criteria import large_final
from fasterai.sparse.sparsifier import Sparsifier

In [ ]:
from fastcore.test import *
import torch.nn as nn
import warnings

def _test_model():
    return nn.Sequential(
        nn.Conv2d(3, 16, 3, padding=1),
        nn.BatchNorm2d(16),
        nn.ReLU(),
        nn.Conv2d(16, 32, 3, padding=1),
        nn.BatchNorm2d(32),
        nn.ReLU(),
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.Linear(32, 10)
    )

model = _test_model()
sample = torch.randn(2, 3, 8, 8)
def _eval(m):
    m.eval()
    with torch.no_grad(): return m(sample).abs().mean().item()

analyzer = SensitivityAnalyzer(model, sample, _eval)

# Invalid compression raises ValueError
with ExceptionExpected(ValueError):
    analyzer.analyze('invalid', 0.5, verbose=False)

# Result structure for sparsity analysis; the level is kept as the fraction it was given
result = analyzer.analyze('sparsity', 0.3, verbose=False)
assert isinstance(result, SensitivityResult)
test_eq(result.compression_level, 0.3)
assert len(result.layers) > 0
assert isinstance(result.layers[0], LayerSensitivity)

# 0.3 really means 30%: the compressed metric moves as much as a 30% sparsify does
_probe = _test_model()
_sp = Sparsifier(_probe, granularity='weight', context='local', criteria=large_final)
_sp.sparsify_layer(_probe[0], 0.3)
test_close((_probe[0].weight == 0).float().mean().item(), 0.3, eps=0.05)

# A percent still works for one release, and warns
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    _pct = SensitivityAnalyzer(_test_model(), sample, _eval).analyze('sparsity', 30, verbose=False)
test_eq(_pct.compression_level, 0.3)
assert any(x.category is FutureWarning for x in w)

# Quantization levels are bit widths, never fractions — and never warn
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    _q = SensitivityAnalyzer(_test_model(), sample, _eval).analyze('quantization', 8, verbose=False)
test_eq(_q.compression_level, 8)
test_eq(_q.compression_type, 'quantization-8bit')
test_eq([x for x in w if 'looks like a percent' in str(x.message)], [])

# Backward-compat: sparsity mode leaves the new pruning fields at defaults
test_eq(result.layers[0].group_id, None)
test_eq(result.layers[0].prunable, True)
test_eq(result.layers[0].group_members, [])

# top() sorted correctly (most sensitive first)
top3 = result.top(3, most_sensitive=True)
for i in range(len(top3)-1):
    assert top3[i].delta >= top3[i+1].delta

# to_layer_targets returns FRACTIONS, per layer name
sched = result.to_layer_targets(model, target=0.5)
assert isinstance(sched, dict)
assert len(sched) == len(result.layers)
for v in sched.values():
    assert 0 <= v <= 0.9, v

# min_ratio is honoured as a fraction, and the targets feed Sparsifier/Pruner directly
sched_min = result.to_layer_targets(model, target=0.5, min_ratio=0.005)
assert min(sched_min.values()) >= 0.005
_target_model = _test_model()
Sparsifier(_target_model, 'weight', 'local', large_final).sparsify_model(
    {k: v for k, v in sched_min.items() if k in dict(_target_model.named_modules())})
test_close((_target_model[0].weight == 0).float().mean().item(), sched_min['0'], eps=0.05)

# The percent keywords are gone — the rename is breaking, on purpose
with ExceptionExpected(TypeError):
    result.to_layer_targets(model, target_pct=50)

# LayerSensitivity.as_dict returns proper dict (now includes group fields)
d = result.layers[0].as_dict()
assert 'name' in d
assert 'delta' in d
assert 'params' in d
assert 'group_id' in d and 'prunable' in d

# SensitivityResult.as_dict works
full_d = result.as_dict()
assert 'compression_type' in full_d
assert 'layers' in full_d

# analyze_sensitivity convenience function works
result2 = analyze_sensitivity(
    _test_model(), sample, _eval,
    compression='sparsity', level=0.2, verbose=False
)
assert isinstance(result2, SensitivityResult)
test_eq(result2.compression_level, 0.2)

# ── Pruning mode: every prunable layer gets a group_id; the result is FAITHFUL ──
from fasterai.prune.pruner import Pruner
from fasterai.core.criteria import large_final

_pm = _test_model()
_psample = torch.randn(2, 3, 8, 8)
_pbaseline = _eval(_pm)
_pres = SensitivityAnalyzer(_pm, _psample, _eval, criteria=large_final).analyze('pruning', 0.5, verbose=False)
_pby = {l.name: l for l in _pres.layers}

# Internal conv ('0') and Linear analyzed; conv layers are prunable with a group_id
assert _pby['0'].group_id is not None
assert _pby['0'].prunable is True
# Output Linear ('8') is not independently prunable -> excluded from robust ranking
assert _pby['8'].prunable is False
assert _pby['8'] not in _pres.top(5, most_sensitive=False)

# Faithfulness: analysis Δ for a layer == really pruning {layer: level} on a fresh copy
import io
def _clone(mdl):
    b = io.BytesIO(); torch.save(mdl, b); b.seek(0); return torch.load(b, weights_only=False)
_c = _clone(_pm)
Pruner(_c, pruning_ratio={'0': 0.5}, context='local', criteria=large_final,
       example_inputs=_psample).prune_model()
_real_delta = _pbaseline - _eval(_c)
test_close(_pby['0'].delta, _real_delta, eps=1e-5)

# layer_types accepts a single type, a tuple, or a list (all equivalent) and filters out Linear
for _lt in (nn.Conv2d, (nn.Conv2d,), [nn.Conv2d]):
    _r = SensitivityAnalyzer(_test_model(), sample, _eval).analyze('pruning', 0.5, layer_types=_lt, verbose=False)
    test_eq({l.layer_type for l in _r.layers}, {'Conv2d'})

In [ ]:
# The fake-quantization formulas are pinned: symmetric/affine x per-channel/per-tensor.
_t = torch.tensor([[-1., 0.5], [2., -0.25]])
test_eq(analyzer._fake_quantize(_t, 8, symmetric=True).tolist(),
        [[-1.0078740119934082, 0.5039370059967041], [2.0, -0.25196850299835205]])
test_eq(analyzer._fake_quantize(_t, 4, symmetric=True, per_channel=True).tolist(),
        [[-1.0, 0.4285714626312256], [2.0, -0.2857142984867096]])
test_eq(analyzer._fake_quantize(_t, 4, symmetric=False).tolist(),
        [[-1.0, 0.4000000059604645], [2.0, -0.20000000298023224]])
test_eq(analyzer._fake_quantize(_t, 8, symmetric=False, per_channel=True).tolist(),
        [[-1.0, 0.5], [2.002941370010376, -0.24705883860588074]])

# A 1-D tensor has no channel axis to quantize along: it falls back to per-tensor
test_eq(analyzer._fake_quantize(torch.tensor([-1., 0.5, 2., -0.25]), 8, per_channel=True).tolist(),
        [-1.0078740119934082, 0.5039370059967041, 2.0, -0.25196850299835205])

In [ ]:
# One pinned analyze() per mode: layer order, params, group ids, prunable flags, deltas.
def _pin_net():
    torch.manual_seed(0)
    return nn.Sequential(
        nn.Conv2d(3, 8, 3), nn.BatchNorm2d(8), nn.ReLU(),
        nn.Conv2d(8, 16, 3), nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Linear(16, 10),
    ).eval()

torch.manual_seed(123)
_pin_batch, _pin_sample = torch.randn(2, 3, 16, 16), torch.randn(2, 3, 16, 16)
def _pin_eval(m):
    m.eval()
    with torch.no_grad(): return m(_pin_batch).mean().item()

_pins = {
    ('sparsity', 0.5): ('sparsity', [None]*3, [True]*3,
        [-0.003372013568878174, -0.010508306324481964, -0.005343683063983917]),
    ('pruning', 0.3): ('pruning', [0, 1, None], [True, True, False],
        [-0.010957978665828705, 0.02224069833755493, 0.0]),
    ('quantization', 8): ('quantization-8bit', [None]*3, [True]*3,
        [1.5787780284881592e-05, 0.00011195987462997437, 0.000150337815284729]),
}
# CPU reductions are thread-order dependent, so the deltas are pinned single-threaded
_prev_threads = torch.get_num_threads()
torch.set_num_threads(1)
try:
    for (_c, _l), (_type, _gids, _prunable, _deltas) in _pins.items():
        _r = SensitivityAnalyzer(_pin_net(), _pin_sample, _pin_eval).analyze(_c, _l, verbose=False)
        test_eq(_r.compression_type, _type)
        test_eq([x.name for x in _r.layers], ['0', '3', '6'])
        test_eq([x.params for x in _r.layers], [216, 1152, 160])
        test_eq([x.group_id for x in _r.layers], _gids)
        test_eq([x.prunable for x in _r.layers], _prunable)
        for _lay in _r.layers:
            test_eq(_lay.delta, _r.baseline_metric - _lay.compressed_metric)
        test_close([x.delta for x in _r.layers], _deltas, eps=1e-6)
finally:
    torch.set_num_threads(_prev_threads)

In [ ]:
#| slow
# Full sensitivity analysis on a larger model
from torchvision.models import resnet18

_model_lg = resnet18(weights=None)
_sample_lg = torch.randn(2, 3, 32, 32)
def _eval_lg(m):
    m.eval()
    with torch.no_grad(): return m(_sample_lg).abs().mean().item()

_result_lg = analyze_sensitivity(_model_lg, _sample_lg, _eval_lg,
                                 compression='sparsity', level=0.3, verbose=False)
assert isinstance(_result_lg, SensitivityResult)
assert len(_result_lg.layers) > 3  # resnet18 has many conv layers

_sched_lg = _result_lg.to_layer_targets(_model_lg, target=0.3)
assert len(_sched_lg) > 3
for v in _sched_lg.values():
    assert 0 <= v <= 0.9, v

# The targets are fractions a Pruner accepts as-is (no rescaling, no warning)
import warnings
with warnings.catch_warnings(record=True) as _w:
    warnings.simplefilter("always")
    _p_lg = Pruner(resnet18(weights=None), {'layer1.0.conv1': _sched_lg['layer1.0.conv1']},
                   'local', large_final, example_inputs=_sample_lg)
test_eq([x for x in _w if 'looks like a percent' in str(x.message)], [])

In [ ]:
#| slow
# Pruning group-sensitivity on ResNet-18 — regression test for the false-robust bug
# (residual-coupled layers used to report Δ=0 "robust" because they could not be pruned in isolation).
import io
from torchvision.models import resnet18
from fasterai.prune.pruner import Pruner
from fasterai.core.criteria import large_final

torch.manual_seed(0)
_rn = resnet18(weights=None).eval()
_s = torch.randn(2, 3, 32, 32)
def _ev(m):
    m.eval()
    with torch.no_grad(): return m(_s).abs().mean().item()
_base = _ev(_rn)

_res = SensitivityAnalyzer(_rn, _s, _ev, criteria=large_final).analyze('pruning', 0.5, verbose=False)
_by = {l.name: l for l in _res.layers}

# Residual-coupled layers (stem conv1, block conv2, downsample) are NO LONGER falsely "robust":
# the bug made their delta EXACTLY 0 (a no-op prune). Now each is prunable and actually moves the metric.
_coupled = ['conv1', 'layer1.0.conv2', 'layer2.0.downsample.0']
_robust_names = {l.name for l in _res.top(5, most_sensitive=False)}
for _name in _coupled:
    assert _name in _by, f"{_name} missing from result"
    assert _by[_name].prunable, f"{_name} should be prunable via its group"
    assert abs(_by[_name].delta) > 1e-6, f"{_name} falsely reports Δ≈0 (the no-op bug)"
    assert _name not in _robust_names, f"{_name} wrongly ranked most-robust"

# Coupled layers share a group_id and therefore an identical delta
assert _by['conv1'].group_id == _by['layer1.0.conv2'].group_id
test_close(_by['conv1'].delta, _by['layer1.0.conv2'].delta, eps=1e-6)

# An internal conv (output feeds only the next conv) is in a DIFFERENT group, prunable, and actually moved
assert _by['layer1.0.conv1'].group_id != _by['conv1'].group_id
assert _by['layer1.0.conv1'].prunable and abs(_by['layer1.0.conv1'].delta) > 1e-6

# Faithfulness: the reported Δ equals really pruning {layer: level} on a fresh copy
def _clone(mdl):
    b = io.BytesIO(); torch.save(mdl, b); b.seek(0); return torch.load(b, weights_only=False)
_c = _clone(_rn)
Pruner(_c, pruning_ratio={'conv1': 0.5}, context='local', criteria=large_final,
       example_inputs=_s).prune_model()
_real = _base - _ev(_c)
test_close(_by['conv1'].delta, _real, eps=1e-5)

# summary() displays the level as a percentage
_res.summary()
test_eq(_res._level_str(), '50.00%')